<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_18_Real_Private_A_B_Information_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from decimal import Decimal
import os, json, hashlib, sqlite3, platform

try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
except ImportError:
    pass
MYDRIVE=Path(os.environ.get('ICHI2027_MYDRIVE','/content/drive/MyDrive')).expanduser().resolve()
assert MYDRIVE.is_dir(), 'Mount the SAME Google Drive used for Phase17, or define ICHI2027_MYDRIVE.'
PROJECT=MYDRIVE/'NeuroFHIR_ICHI2027'
OUT=PROJECT/'phase18_outputs'
PRIVATE=OUT/'_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True,exist_ok=True)
SHAREABLE=OUT/'ICHI2027_Phase18_SHAREABLE.json'
PRIVATE_MANIFEST=PRIVATE/'ICHI2027_Phase18_PRIVATE_AB_MANIFEST.json'
DB=PRIVATE/'ICHI2027_Phase18_REAL_AB_CANDIDATE_STAGING_PRIVATE.sqlite'
TEMP_DB=PRIVATE/'ICHI2027_Phase18_REAL_AB_CANDIDATE_STAGING_PRIVATE.sqlite.part'
FEATURES=('LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV')
BIO_FEATURES=tuple(f for f in FEATURES if f!='ICV')
STATES={'TRUSTED','CAUTION','REVIEW_REQUIRED'}
FIELD_SYSTEM='https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-field'
PAIR_PREFIX='Research-only TRACE Q90 longitudinal subject-pair reliability for '
PACKAGE_TEXT='Research-only descriptive TRACE subject-pair package classification'
EXPECTED={'bundles':258,'persons':258,'images':516,'numeric':4128,'feature_states':1806,'packages':258}

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def numeric_text(value):
    assert isinstance(value,(float,int)) and not isinstance(value,bool),'Expected real FHIR JSON numeric value'
    number=Decimal(str(value))
    assert number.is_finite(),'Non-finite measurement encountered'
    return format(number.normalize(),'f')

def utcnow():return datetime.now(timezone.utc).isoformat()
print('Phase18: genuine real-FHIR two-arm private staging; no browser file uploads/downloads.')


Mounted at /content/drive
Phase18: genuine real-FHIR two-arm private staging; no browser file uploads/downloads.


In [2]:
public17_path=Path(os.environ.get('ICHI2027_PHASE17_JSON',str(PROJECT/'phase17_outputs'/'ICHI2027_Phase17_SHAREABLE.json')))
private17_path=Path(os.environ.get('ICHI2027_PHASE17_PRIVATE_MANIFEST',str(PROJECT/'phase17_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase17_PRIVATE_LINKAGE_MANIFEST.json')))
assert public17_path.is_file(),'Phase17 shareable JSON missing. Save it to phase17_outputs/ in the original Drive.'
assert private17_path.is_file(),'Phase17 PRIVATE manifest missing. Do not upload it publicly; restore from original Drive.'
p17=json.loads(public17_path.read_text(encoding='utf-8'))
m17=json.loads(private17_path.read_text(encoding='utf-8'))
assert p17['phase']=='17_real_pair_level_TRACE_to_FHIR_linkage'
assert p17['fhir_r4']['official_schema_validation']=='PASS'
assert p17['fhir_r4']['validated_resource_counts']=={'Bundle':258,'Patient':258,'ImagingStudy':516,'Observation':6192}
assert p17['private_pair_linkage_audit']['biological_feature_states']==258*7
assert p17['fhir_r4']['all_numeric_decimal_roundtrips_exact'] is True
assert p17['exact_private_evidence_reconciliation']['original_phase09_private_HMAC_key_reused'] is True
assert m17['phase']=='17_real_pair_level_TRACE_to_FHIR_linkage'
assert m17['run_utc']==p17['run_utc']
assert m17['original_source_notebook_sha256']==p17['reused_original_20b_method']['notebook_sha256']
FHIR_PATH=Path(os.environ.get('ICHI2027_PHASE17_PRIVATE_FHIR',m17['private_FHIR_NDJSON']))
if not FHIR_PATH.is_file():
    FHIR_PATH=PROJECT/'phase17_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase17_Real_TRACE_Pair_Linked_FHIR_R4_PRIVATE.ndjson'
assert FHIR_PATH.is_file(),'Original Phase17 private real FHIR NDJSON missing. Restore from Drive; never reconstruct from shareable counts.'
assert sha256_file(FHIR_PATH)==m17['private_FHIR_NDJSON_sha256'],'Private FHIR file does not match Phase17 signed private manifest.'
assert m17['fhir_r4_schema_sha256']==p17['fhir_r4']['official_pinned_archive_sha256']
print('Phase17 public/private same run; actual FHIR NDJSON SHA-256 verified.')


Phase17 public/private same run; actual FHIR NDJSON SHA-256 verified.


In [3]:
# The arm definitions are explicit and frozen before processing any participant record.
# A is a lossless PRIVATE RESEARCH CANDIDATE of the fields specified below, not official MI-CDM.
# B retains person, two timepoints, feature and numeric value; it intentionally omits
# ImagingStudy links, original source identifier provenance and derived TRACE pair assessments.
# Units, DICOM UIDs and OMOP concepts are absent/unverified in the source and not imputed in either arm.
ARM_CONTRACT={
 'A':{'name':'provenance_preserving_private_research_candidate',
      'retains':['person','timepoint','observation_date','feature','numeric_value','FHIR_observation_key',
                 'source_ImagingStudy_link','raw_ADNI_identifier_fields_as_unverified_source_labels',
                 'regenerated_TRACE_pair_feature_state','both_source_volume_links_per_pair_state',
                 'regenerated_TRACE_pair_package_state','seven_feature_state_package_links',
                 'original_20B_replay_notebook_checksum','regenerated_calibration_CSV_checksum'],
      'explicitly_unverified':['numeric_units','DICOM_study_series_UIDs','OMOP_concepts','official_MI_CDM_target']},
 'B':{'name':'deliberate_value_only_private_research_ablation',
      'retains':['person','timepoint','observation_date','feature','numeric_value','FHIR_observation_key'],
      'intentionally_omits':['ImagingStudy_table','source_image_identifier_fields','source_image_link',
                            'regenerated_TRACE_pair_feature_state','BL_M12_PAIR_derivedFrom_links',
                            'TRACE_pair_package_state','frozen_replay_provenance'],
      'explicitly_unverified':['numeric_units','DICOM_study_series_UIDs','OMOP_concepts','official_MI_CDM_target']}}
contract_sha256=hashlib.sha256(json.dumps(ARM_CONTRACT,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('A/B arm contract frozen locally. This is a design-defined ablation, not yet a four-arm effectiveness claim.')


A/B arm contract frozen locally. This is a design-defined ablation, not yet a four-arm effectiveness claim.


In [4]:
SCHEMA='''
CREATE TABLE A_PERSON (person_key TEXT PRIMARY KEY);
CREATE TABLE A_IMAGE (
 image_key TEXT PRIMARY KEY,person_key TEXT NOT NULL REFERENCES A_PERSON(person_key),
 image_date TEXT NOT NULL,source_identifiers_JSON TEXT NOT NULL);
CREATE TABLE A_NUMERIC (
 observation_key TEXT PRIMARY KEY,person_key TEXT NOT NULL REFERENCES A_PERSON(person_key),
 image_key TEXT NOT NULL REFERENCES A_IMAGE(image_key),timepoint TEXT NOT NULL CHECK(timepoint IN ('BL','M12')),
 feature TEXT NOT NULL,observation_date TEXT NOT NULL,value_decimal TEXT NOT NULL,
 source_anatomy_text TEXT);
CREATE TABLE A_PAIR_STATE (
 pair_key TEXT PRIMARY KEY,person_key TEXT NOT NULL REFERENCES A_PERSON(person_key),
 feature TEXT NOT NULL,state TEXT NOT NULL CHECK(state IN ('TRUSTED','CAUTION','REVIEW_REQUIRED')),
 baseline_observation_key TEXT NOT NULL UNIQUE REFERENCES A_NUMERIC(observation_key),
 month12_observation_key TEXT NOT NULL UNIQUE REFERENCES A_NUMERIC(observation_key),
 original_method_SHA256 TEXT NOT NULL,regenerated_calibration_CSV_SHA256 TEXT NOT NULL,
 method_note TEXT);
CREATE TABLE A_PACKAGE_STATE (
 package_key TEXT PRIMARY KEY,person_key TEXT NOT NULL UNIQUE REFERENCES A_PERSON(person_key),
 state TEXT NOT NULL CHECK(state IN ('TRUSTED','CAUTION','REVIEW_REQUIRED')));
CREATE TABLE A_PACKAGE_MEMBER (
 package_key TEXT NOT NULL REFERENCES A_PACKAGE_STATE(package_key),
 pair_key TEXT NOT NULL UNIQUE REFERENCES A_PAIR_STATE(pair_key),
 PRIMARY KEY(package_key,pair_key));
CREATE TABLE B_PERSON (person_key TEXT PRIMARY KEY);
CREATE TABLE B_NUMERIC (
 observation_key TEXT PRIMARY KEY,person_key TEXT NOT NULL REFERENCES B_PERSON(person_key),
 timepoint TEXT NOT NULL CHECK(timepoint IN ('BL','M12')),
 feature TEXT NOT NULL,observation_date TEXT NOT NULL,value_decimal TEXT NOT NULL);
CREATE INDEX idx_A_person_feature_visit ON A_NUMERIC(person_key,feature,timepoint);
CREATE INDEX idx_B_person_feature_visit ON B_NUMERIC(person_key,feature,timepoint);
'''
SCHEMA_SHA256=hashlib.sha256(SCHEMA.encode('utf-8')).hexdigest()
if TEMP_DB.exists():TEMP_DB.unlink()
con=sqlite3.connect(TEMP_DB)
con.execute('PRAGMA foreign_keys=ON')
con.executescript(SCHEMA)
count=Counter()
seen_global=set()
source_package_states=Counter()
source_feature_states=Counter()
try:
    with FHIR_PATH.open('r',encoding='utf-8') as stream:
        for raw_line in stream:
            if not raw_line.strip():continue
            bundle=json.loads(raw_line)
            assert bundle.get('resourceType')=='Bundle' and bundle.get('type')=='collection'
            entries=bundle.get('entry',[])
            assert len(entries)==27,'Unexpected Phase17 bundle size: STOP rather than omit any real information.'
            index={e['fullUrl']:e['resource'] for e in entries}
            assert len(index)==len(entries)
            assert not(set(index)&seen_global),'Repeated FHIR fullUrl across separate patient bundles.'
            seen_global.update(index)
            people=[(k,r) for k,r in index.items() if r['resourceType']=='Patient']
            images=[(k,r) for k,r in index.items() if r['resourceType']=='ImagingStudy']
            observations=[(k,r) for k,r in index.items() if r['resourceType']=='Observation']
            assert len(people)==1 and len(images)==2 and len(observations)==24
            person_key=people[0][0]
            assert person_key.startswith('urn:uuid:')
            count['bundles']+=1
            con.execute('INSERT INTO A_PERSON VALUES (?)',(person_key,))
            con.execute('INSERT INTO B_PERSON VALUES (?)',(person_key,))
            count['persons']+=1
            image_dates={}
            for image_key,img in images:
                assert img['subject']['reference']==person_key
                assert img.get('status')=='unknown'
                day=img.get('started')
                assert day and len(day)>=10
                source_ids=img.get('identifier',[])
                assert len(source_ids)==2 and {x.get('system') for x in source_ids}=={
                    'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-imageuid',
                    'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-loniuid'},'Original two source-labelled identifiers missing or modified.'
                image_dates[image_key]=day
                con.execute('INSERT INTO A_IMAGE VALUES (?,?,?,?)',
                            (image_key,person_key,day,json.dumps(source_ids,sort_keys=True)))
                count['images']+=1
            numeric_obs={}
            pairs={}
            package=[]
            for obs_key,obs in observations:
                assert obs['subject']['reference']==person_key
                id_fields=[x.get('value','') for x in obs.get('identifier',[]) if x.get('system')==FIELD_SYSTEM]
                if 'valueQuantity' in obs:
                    assert len(id_fields)==1 and '_' in id_fields[0]
                    tp,feature=id_fields[0].split('_',1)
                    assert tp in ('BL','M12') and feature in FEATURES
                    assert 'unit' not in obs['valueQuantity'] and 'code' not in obs['valueQuantity'],'An unexpected verified unit appears; update the predeclared arm contract before proceeding.'
                    refs=[x['reference'] for x in obs.get('derivedFrom',[])]
                    assert len(refs)==1 and refs[0] in image_dates
                    assert obs['effectiveDateTime']==image_dates[refs[0]]
                    assert (tp,feature) not in numeric_obs
                    value=numeric_text(obs['valueQuantity']['value'])
                    anatomy=obs.get('bodySite',{}).get('text','')
                    con.execute('INSERT INTO A_NUMERIC VALUES (?,?,?,?,?,?,?,?)',
                                (obs_key,person_key,refs[0],tp,feature,obs['effectiveDateTime'],value,anatomy))
                    con.execute('INSERT INTO B_NUMERIC VALUES (?,?,?,?,?,?)',
                                (obs_key,person_key,tp,feature,obs['effectiveDateTime'],value))
                    numeric_obs[(tp,feature)]=(obs_key,value)
                    count['numeric']+=1
                else:
                    code=obs.get('code',{}).get('text','')
                    if code.startswith(PAIR_PREFIX):
                        feature=code[len(PAIR_PREFIX):]
                        assert feature in BIO_FEATURES and feature not in pairs
                        state=obs.get('valueCodeableConcept',{}).get('text')
                        assert state in STATES
                        refs=[x['reference'] for x in obs.get('derivedFrom',[])]
                        assert len(refs)==2 and obs.get('effectivePeriod',{}).get('start') and obs.get('effectivePeriod',{}).get('end')
                        pairs[feature]=(obs_key,obs,state,refs)
                        source_feature_states[state]+=1
                    elif code==PACKAGE_TEXT:
                        package.append((obs_key,obs))
                    else:raise AssertionError('Unexpected Phase17 nonnumeric observation: refuse silent omission.')
            assert set(numeric_obs)=={(tp,f) for tp in ('BL','M12') for f in FEATURES}
            assert set(pairs)==set(BIO_FEATURES) and len(package)==1
            assert len({img for img in image_dates})==2
            for feature,(pair_key,obs,state,refs) in pairs.items():
                assert refs==[numeric_obs[('BL',feature)][0],numeric_obs[('M12',feature)][0]]
                assert obs['effectivePeriod']['start']==index[refs[0]]['effectiveDateTime']
                assert obs['effectivePeriod']['end']==index[refs[1]]['effectiveDateTime']
                assert 'Original corrected 20B code replayed' in obs.get('method',{}).get('text','')
                con.execute('INSERT INTO A_PAIR_STATE VALUES (?,?,?,?,?,?,?,?,?)',
                            (pair_key,person_key,feature,state,refs[0],refs[1],
                             m17['original_source_notebook_sha256'],m17['original_replay_calibration_csv_sha256_PRIVATE'],
                             obs['method']['text']))
                count['feature_states']+=1
            package_key,pkg=package[0]
            pkg_state=pkg.get('valueCodeableConcept',{}).get('text')
            assert pkg_state in STATES
            assert pkg['effectivePeriod']['start']==index[numeric_obs[('BL','LHIPPOC')][0]]['effectiveDateTime']
            assert pkg['effectivePeriod']['end']==index[numeric_obs[('M12','LHIPPOC')][0]]['effectiveDateTime']
            refs=[x['reference'] for x in pkg.get('derivedFrom',[])]
            assert len(refs)==7 and set(refs)=={v[0] for v in pairs.values()}
            con.execute('INSERT INTO A_PACKAGE_STATE VALUES (?,?,?)',(package_key,person_key,pkg_state))
            for ref in refs:con.execute('INSERT INTO A_PACKAGE_MEMBER VALUES (?,?)',(package_key,ref))
            source_package_states[pkg_state]+=1
            count['packages']+=1
    assert all(count.get(k)==v for k,v in EXPECTED.items()),f'FHIR resource counts changed: {dict(count)}'
    assert sum(source_feature_states.values())==1806 and sum(source_package_states.values())==258
    con.commit()
except Exception:
    con.rollback()
    con.close()
    if TEMP_DB.exists():TEMP_DB.unlink()
    raise
print('Actual original Phase17 FHIR transformed to arm A and arm B private staging, with exact source reference checks.')


Actual original Phase17 FHIR transformed to arm A and arm B private staging, with exact source reference checks.


In [5]:
# Check the two different candidate representations against the exact same source numeric FHIR events.
def n(table):return con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
assert all(n(k)==v for k,v in {
    'A_PERSON':258,'A_IMAGE':516,'A_NUMERIC':4128,'A_PAIR_STATE':1806,
    'A_PACKAGE_STATE':258,'A_PACKAGE_MEMBER':1806,'B_PERSON':258,'B_NUMERIC':4128}.items())
fk_errors=con.execute('PRAGMA foreign_key_check').fetchall()
assert not fk_errors
# A and B have identical numeric values, dates and membership. The omitted data exist only in A by predeclared design.
numeric_diff=con.execute('''SELECT COUNT(*) FROM A_NUMERIC a LEFT JOIN B_NUMERIC b USING(observation_key)
  WHERE b.observation_key IS NULL OR a.person_key!=b.person_key OR a.feature!=b.feature
    OR a.timepoint!=b.timepoint OR a.observation_date!=b.observation_date
    OR a.value_decimal!=b.value_decimal''').fetchone()[0]
numeric_diff_b=con.execute('SELECT COUNT(*) FROM B_NUMERIC b LEFT JOIN A_NUMERIC a USING(observation_key) WHERE a.observation_key IS NULL').fetchone()[0]
assert numeric_diff==numeric_diff_b==0
image_link_errors=con.execute('''SELECT COUNT(*) FROM A_NUMERIC n JOIN A_IMAGE i ON n.image_key=i.image_key
 WHERE n.person_key!=i.person_key OR n.observation_date!=i.image_date''').fetchone()[0]
assert image_link_errors==0
pair_error=con.execute('''SELECT COUNT(*) FROM A_PAIR_STATE s
 JOIN A_NUMERIC bl ON bl.observation_key=s.baseline_observation_key
 JOIN A_NUMERIC m ON m.observation_key=s.month12_observation_key
 WHERE bl.person_key!=s.person_key OR m.person_key!=s.person_key OR bl.feature!=s.feature
 OR m.feature!=s.feature OR bl.timepoint!='BL' OR m.timepoint!='M12' ''').fetchone()[0]
assert pair_error==0
package_member_error=con.execute('''SELECT COUNT(*) FROM A_PACKAGE_MEMBER pm
 JOIN A_PACKAGE_STATE p ON p.package_key=pm.package_key
 JOIN A_PAIR_STATE s ON s.pair_key=pm.pair_key WHERE p.person_key!=s.person_key''').fetchone()[0]
assert package_member_error==0
package_degree_error=con.execute('''SELECT COUNT(*) FROM (SELECT package_key,COUNT(*) cnt FROM A_PACKAGE_MEMBER
 GROUP BY package_key HAVING cnt != 7)''').fetchone()[0]
assert package_degree_error==0

def structural_cohort(table):
    return {r[0] for r in con.execute(f'''SELECT person_key FROM {table}
      WHERE feature IN ('LHIPPOC','RHIPPOC')
      GROUP BY person_key HAVING COUNT(DISTINCT timepoint||':'||feature)=4''')}
cohort_A=structural_cohort('A_NUMERIC')
cohort_B=structural_cohort('B_NUMERIC')
assert cohort_A==cohort_B and len(cohort_A)==258
# Verify the intentional design is explicit in actual table schemas, not based on a missing-value proxy.
cols_B={x[1] for x in con.execute('PRAGMA table_info(B_NUMERIC)').fetchall()}
assert cols_B=={'observation_key','person_key','timepoint','feature','observation_date','value_decimal'}
assert 'image_key' not in cols_B and 'state' not in cols_B
# Repeated inserts are idempotent and must not overwrite the existing values.
first_B=con.execute('SELECT observation_key,person_key,timepoint,feature,observation_date,value_decimal FROM B_NUMERIC LIMIT 1').fetchone()
assert first_B is not None
before=n('B_NUMERIC')
con.execute('INSERT OR IGNORE INTO B_NUMERIC VALUES (?,?,?,?,?,?)',first_B)
assert n('B_NUMERIC')==before
duplicate_additional_rows=n('B_NUMERIC')-before
con.commit()
assert con.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
con.close()
os.replace(TEMP_DB,DB)
print('Real A/B integrity PASS: exact numeric parity, two-arm structural cohort parity, all 7 pair links per package and zero FK violations.')


Real A/B integrity PASS: exact numeric parity, two-arm structural cohort parity, all 7 pair links per package and zero FK violations.


In [6]:
# Private artifacts and checksums remain in Drive. The public report contains no row keys or private-file digests.
private_manifest={
 'project':p17['project'],'phase':'18_real_private_two_arm_A_B_ablation',
 'run_utc':utcnow(),'original_phase17_run_utc':p17['run_utc'],
 'phase17_public_json_SHA256':sha256_file(public17_path),
 'phase17_PRIVATE_manifest_SHA256':sha256_file(private17_path),
 'source_private_FHIR_SHA256':sha256_file(FHIR_PATH),
 'source_phase17_private_fhir_SHA256_approved':m17['private_FHIR_NDJSON_sha256'],
 'original_TRACE_20b_notebook_sha256':m17['original_source_notebook_sha256'],
 'regenerated_calibration_SHA256':m17['original_replay_calibration_csv_sha256_PRIVATE'],
 'arm_contract':ARM_CONTRACT,'arm_contract_sha256':contract_sha256,
 'derived_research_candidate_schema_SHA256':SCHEMA_SHA256,
 'private_A_B_SQLite_file':str(DB),'private_A_B_SQLite_SHA256':sha256_file(DB),
 'not_official_MICDM':True,'not_frozen_generic_TFL_v15_comparator':True,
 'no_original_trace_historical_output_claim':True,
 'source_approved_measurement_units':False,'source_approved_DICOM_UID_semantics':False,
 'source_approved_OMOP_concepts':False}
PRIVATE_MANIFEST.write_text(json.dumps(private_manifest,indent=2,allow_nan=False),encoding='utf-8')
print('PRIVATE A/B research candidate, source checksums, calibration provenance and contract saved to your original Drive.')


PRIVATE A/B research candidate, source checksums, calibration provenance and contract saved to your original Drive.


In [7]:
# Descriptive audit of deliberately removed information, not an algorithm superiority score.
with sqlite3.connect(DB) as c:
    a_links=c.execute('SELECT COUNT(*) FROM A_NUMERIC WHERE image_key IS NOT NULL').fetchone()[0]
    a_pair_links=c.execute('SELECT COUNT(*) FROM A_PAIR_STATE WHERE baseline_observation_key IS NOT NULL AND month12_observation_key IS NOT NULL').fetchone()[0]
    a_pkg_links=c.execute('SELECT COUNT(*) FROM A_PACKAGE_MEMBER').fetchone()[0]
    a_source_labels=c.execute('SELECT COUNT(*) FROM A_IMAGE WHERE source_identifiers_JSON IS NOT NULL').fetchone()[0]
    a_calib_links=c.execute('SELECT COUNT(*) FROM A_PAIR_STATE WHERE regenerated_calibration_CSV_SHA256 IS NOT NULL').fetchone()[0]
    again={name:c.execute('SELECT COUNT(*) FROM '+name).fetchone()[0] for name in ('A_PERSON','A_IMAGE','A_NUMERIC','A_PAIR_STATE','A_PACKAGE_STATE','A_PACKAGE_MEMBER','B_PERSON','B_NUMERIC')}
assert a_links==4128 and a_pair_links==1806 and a_pkg_links==1806 and a_source_labels==516 and a_calib_links==1806
assert again=={'A_PERSON':258,'A_IMAGE':516,'A_NUMERIC':4128,'A_PAIR_STATE':1806,'A_PACKAGE_STATE':258,'A_PACKAGE_MEMBER':1806,'B_PERSON':258,'B_NUMERIC':4128}
public={
 'project':p17['project'],'phase':'18_real_private_two_arm_A_B_ablation',
 'run_utc':private_manifest['run_utc'],
 'result_status':'ACTUAL_REAL_SOURCE_TWO_ARM_DESCRIPTIVE_PRIVATE_CANDIDATE_ABLATION_ONLY_NOT_OFFICIAL_MICDM_NOT_COMPLETE_FOUR_ARM',
 'privacy':'Aggregate-only; all source observations, images, participant linkage, private source provenance, database and artifact hashes remain in private Drive.',
 'phase17_public_json_sha256':sha256_file(public17_path),
 'source_inherited_validation':{'pinned_FHIR_R4_validation':'PASS_IN_PHASE17','phase17_private_FHIR_SHA256_verified':True,
     'replayed_TRACE_20b_not_historical_freeze':True,'original_TRACE_sidecar_is_subject_pair_not_scan_level':True},
 'arm_contract_sha256':contract_sha256,
 'real_input_counts':{'source_FHIR_bundles':count['bundles'],'patients':count['persons'],
     'source_ImagingStudy':count['images'],'source_numeric_Observations':count['numeric'],
     'regenerated_TRACE_pair_feature_Observations':count['feature_states'],
     'regenerated_TRACE_pair_package_Observations':count['packages']},
 'actual_arm_A':{'type':'PRIVATE_FIDELITY_PRESERVING_RESEARCH_CANDIDATE_NOT_MICDM',
     'persons':again['A_PERSON'],'image_events':again['A_IMAGE'],'numeric_events':again['A_NUMERIC'],
     'pair_feature_states':again['A_PAIR_STATE'],'pair_package_states':again['A_PACKAGE_STATE'],
     'source_numeric_to_image_links':a_links,'source_image_records_with_unverified_source_identifier_fields':a_source_labels,
     'pair_feature_states_with_both_original_numeric_links':a_pair_links,
     'pair_package_to_feature_links':a_pkg_links,'regenerated_calibration_provenance_links':a_calib_links},
 'actual_arm_B':{'type':'DELIBERATE_VALUE_ONLY_ABLATION_RESEARCH_CANDIDATE_NOT_MICDM',
     'persons':again['B_PERSON'],'numeric_events':again['B_NUMERIC'],
     'intentionally_retained':['person','timepoint','feature','date','original_numeric_value','FHIR_observation_key'],
     'intentionally_removed':['source_ImagingStudy_and_source_image_identifiers','numeric_to_image_links',
         'regenerated_TRACE_pair_feature_states_and_links','regenerated_TRACE_pair_package_states_and_links',
         'replayed_calibration_provenance']},
 'direct_A_vs_B_invariants':{'numeric_event_key_and_exact_Decimal_value_mismatches':numeric_diff+numeric_diff_b,
     'foreign_key_violations':len(fk_errors),'A_numeric_image_temporal_or_person_mismatches':image_link_errors,
     'A_pair_2_observation_link_errors':pair_error,'A_package_feature_person_link_errors':package_member_error,
     'A_package_degree_not_7_errors':package_degree_error,'structural_source_eligible_persons_A':len(cohort_A),
     'structural_source_eligible_persons_B':len(cohort_B),
     'structural_cohort_membership_symmetric_difference':len(cohort_A^cohort_B),
     'arm_B_duplicate_reimport_additional_rows':duplicate_additional_rows},
 'interpretation':'Only the intentionally specified data elements differ. Identical numeric and structural cohort values are a software parity check, not proof of clinical equivalence or TFL effectiveness. Missing units were absent at input, not lost by arm B.',
 'limitations':{'exact_ADNI_original_export_and_unit_attestation':False,
     'DICOM_StudyInstanceUID_and_SeriesInstanceUID_attestation':False,
     'original_REPEAT_acquisition_type_independent_provenance':False,
     'full_Athena_snapshot_and_approved_concepts':False,
     'official_MICDM_load':False,'frozen_generic_TFL_v15_adapted_to_current_real_MRI':False,
     'MRI_specific_TFL_D_executed':False,'four_arm_inferential_comparison':False,
     'baseline_MCI_case_definition_verified':False},
 'four_arm_comparator_scores':None,
 'official_MICDM_load_results':None,
 'next_gate':'PIN_AND_RUN_FROZEN_GENERIC_TFL_v15_ON_AUDITED_REAL_MRI_WITH_EXPLICIT_MAPPING_CONTRACT; OBTAIN_RAW_ADNI_EXPORT_AND_ATHENA_FOR_OFFICIAL_MICDM; IMPLEMENT_INDEPENDENT_MRI_TFL_D_WITHOUT_POSTHOC_TUNING'}
assert public['actual_arm_A']['pair_feature_states']==258*7
assert public['direct_A_vs_B_invariants']['numeric_event_key_and_exact_Decimal_value_mismatches']==0
assert public['direct_A_vs_B_invariants']['structural_cohort_membership_symmetric_difference']==0
SHAREABLE.write_text(json.dumps(public,indent=2,allow_nan=False),encoding='utf-8')
print(json.dumps({'phase':public['phase'],'status':public['result_status'],
 'arm_A_counts':public['actual_arm_A'],'arm_B_counts':{'persons':again['B_PERSON'],'numeric':again['B_NUMERIC']},
 'numeric_mismatches':numeric_diff+numeric_diff_b,'next_gate':public['next_gate']},indent=2))


{
  "phase": "18_real_private_two_arm_A_B_ablation",
  "status": "ACTUAL_REAL_SOURCE_TWO_ARM_DESCRIPTIVE_PRIVATE_CANDIDATE_ABLATION_ONLY_NOT_OFFICIAL_MICDM_NOT_COMPLETE_FOUR_ARM",
  "arm_A_counts": {
    "type": "PRIVATE_FIDELITY_PRESERVING_RESEARCH_CANDIDATE_NOT_MICDM",
    "persons": 258,
    "image_events": 516,
    "numeric_events": 4128,
    "pair_feature_states": 1806,
    "pair_package_states": 258,
    "source_numeric_to_image_links": 4128,
    "source_image_records_with_unverified_source_identifier_fields": 516,
    "pair_feature_states_with_both_original_numeric_links": 1806,
    "pair_package_to_feature_links": 1806,
    "regenerated_calibration_provenance_links": 1806
  },
  "arm_B_counts": {
    "persons": 258,
    "numeric": 4128
  },
  "numeric_mismatches": 0,
  "next_gate": "PIN_AND_RUN_FROZEN_GENERIC_TFL_v15_ON_AUDITED_REAL_MRI_WITH_EXPLICIT_MAPPING_CONTRACT; OBTAIN_RAW_ADNI_EXPORT_AND_ATHENA_FOR_OFFICIAL_MICDM; IMPLEMENT_INDEPENDENT_MRI_TFL_D_WITHOUT_POSTHOC_TUNING"
}

In [8]:
assert SHAREABLE.is_file() and PRIVATE_MANIFEST.is_file() and DB.is_file()
assert json.loads(SHAREABLE.read_text())['four_arm_comparator_scores'] is None
print('Phase18 shareable JSON saved to Google Drive:',SHAREABLE)
print('Download ONLY this aggregate report through drive.google.com, then upload it here.')
print('Keep the _PRIVATE_DO_NOT_SHARE/ folder, original participant FHIR and A/B SQLite private.')


Phase18 shareable JSON saved to Google Drive: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase18_outputs/ICHI2027_Phase18_SHAREABLE.json
Download ONLY this aggregate report through drive.google.com, then upload it here.
Keep the _PRIVATE_DO_NOT_SHARE/ folder, original participant FHIR and A/B SQLite private.
